In [0]:
%sql
CREATE DATABASE IF NOT EXISTS delta_training_db;
USE DATABASE delta_training_db;

In [0]:
%python
customers_data = [
    (1, "Aarav Sharma", "Hyderabad", "Gold", 25000),
    (2, "Priya Reddy", "Bengaluru", "Silver", 18000),
    (3, "Rohan Mehta", "Mumbai", "Gold", 32000),
    (4, "Sneha Iyer", "Chennai", "Bronze", 12000),
    (5, "Kiran Patel", "Hyderabad", "Silver", 22000),
    (6, "Ananya Das", "Kolkata", "Gold", 41000),
    (7, "Vikram Singh", "Delhi", "Bronze", 9000),
    (8, "Meera Nair", "Bengaluru", "Silver", 26000)
]

columns = ["customer_id", "customer_name", "city", "membership", "total_spend"]

customers_df = spark.createDataFrame(customers_data, columns)

display(customers_df)

customer_id,customer_name,city,membership,total_spend
1,Aarav Sharma,Hyderabad,Gold,25000
2,Priya Reddy,Bengaluru,Silver,18000
3,Rohan Mehta,Mumbai,Gold,32000
4,Sneha Iyer,Chennai,Bronze,12000
5,Kiran Patel,Hyderabad,Silver,22000
6,Ananya Das,Kolkata,Gold,41000
7,Vikram Singh,Delhi,Bronze,9000
8,Meera Nair,Bengaluru,Silver,26000


CREATING TABLE USING **DELTA FORMAT** IN SQL

In [0]:
%sql
CREATE OR REPLACE TABLE customers_delta_sql(
  customer_id INT,
  customer_name STRING,
  city STRING,
  membership STRING,
  total_spend INT
)
USING DELTA;

In [0]:
%sql
INSERT INTO customers_delta_sql VALUES
(1, 'John Doe', 'New York', 'Gold', 1000),
(2, 'Jane Smith', 'Los Angeles', 'Silver', 500),
(3, 'Mike Johnson', 'Chicago', 'Bronze', 200);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
SELECT * FROM customers_delta_sql;

customer_id,customer_name,city,membership,total_spend
1,John Doe,New York,Gold,1000
2,Jane Smith,Los Angeles,Silver,500
3,Mike Johnson,Chicago,Bronze,200
1,John Doe,New York,Gold,1000
2,Jane Smith,Los Angeles,Silver,500
3,Mike Johnson,Chicago,Bronze,200


**A delta table using data frame**

In [0]:
%python
customers_df.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable("customers_delta_df")

In [0]:
%sql
SELECT * FROM customers_delta_sql;

customer_id,customer_name,city,membership,total_spend
1,John Doe,New York,Gold,1000
2,Jane Smith,Los Angeles,Silver,500
3,Mike Johnson,Chicago,Bronze,200
1,John Doe,New York,Gold,1000
2,Jane Smith,Los Angeles,Silver,500
3,Mike Johnson,Chicago,Bronze,200


In hybrid you can create a file inside the file storage because this is serverless you can't go ahead and create it inside the file storage so these below  2 code blocks will not get executed in these serverless environment

In [0]:
%python
delta_path = "/FileStore/delta/customers_path_table"

customers_df.write\
    .format("delta")\
    .mode("overwrite")\
    .save(delta_path)

In [0]:
%python
path_df = spark.read.format("delta").load(delta_path)
display(path_df)

In [0]:
%sql
CREATE OR REPLACE TABLE retail_orders
(
order_id INT,
customer_name STRING,
city STRING,
product STRING,
quantity INT,
price INT,
order_status STRING
)
USING DELTA;

In [0]:
%sql
INSERT INTO retail_orders VALUES
(101,'Amit Sharma','Hyderabad','Laptop',1,75000,'Placed'),
(102,'Priya Reddy','Bangalore','Mobile',2,30000,'Placed'),
(103,'Rohit Mehta','Mumbai','Headphones',3,2000,'Shipped'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Placed'),
(105,'Karan Patel','Ahmedabad','Tablet',2,25000,'Cancelled'),
(106,'Ananya Das','Kolkata','Mobile',1,28000,'Placed');

num_affected_rows,num_inserted_rows
6,6


Creating new data for **temporary view**

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW new_orders AS
SELECT * FROM VALUES
(102,'Priya Reddy','Bangalore','Mobile',4,30000,'Delivered'),
(104,'Sneha Iyer','Chennai','Laptop',1,72000,'Shipped'),
(109,'Rahul Verma','Mumbai','Tablet',2,26000,'Placed'),
(110,'Divya Menon','Kochi','Mobile',1,27000,'Placed')
AS new_orders(order_id,customer_name,city,product,quantity,price,order_status);

If order exists -> **UPDATE**
If not exists -> **INSERT**

**Merge**

In [0]:
%sql

MERGE INTO retail_orders AS target

USING new_orders AS source

ON target.order_id = source.order_id
 
WHEN MATCHED THEN

UPDATE SET

target.customer_name = source.customer_name,

target.city = source.city,

target.product = source.product,

target.quantity = source.quantity,

target.price = source.price,

target.order_status = source.order_status
 
WHEN NOT MATCHED THEN

INSERT (

order_id,

customer_name,

city,

product,

quantity,

price,

order_status

)
 
VALUES (

source.order_id,

source.customer_name,

source.city,

source.product,

source.quantity,

source.price,

source.order_status

);
 

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
4,2,0,2


Doing the **upsert**

Insert 2 new orders.
Update price of product Laptop to 78000.
Delete orders where quantity = 1.
Create temp table incoming_orders.
Merge incoming_orders into retail_orders.

In [0]:
%sql
INSERT INTO retail_orders VALUES
(101, 'Arun', 'Chennai', 'Laptop', 2, 75000, 'PLACED'),
(102, 'Meena', 'Coimbatore', 'Mouse', 1, 500, 'PLACED');

num_affected_rows,num_inserted_rows
2,2


In [0]:
%sql
UPDATE retail_orders
SET price = 78000
WHERE product = 'Laptop';

num_affected_rows
3


In [0]:
%sql
DELETE FROM retail_orders
WHERE quantity = 1;

num_affected_rows
5


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW incoming_orders AS
SELECT * FROM VALUES
(101, 'Arun Kumar', 'Chennai', 'Laptop', 3, 78000, 'UPDATED'),
(103, 'Priya', 'Madurai', 'Keyboard', 2, 1500, 'PLACED')
AS incoming_orders
(order_id, customer_name, city, product, quantity, price, order_status);

In [0]:
%sql
MERGE INTO retail_orders AS target
USING incoming_orders AS source
ON target.order_id = source.order_id

WHEN MATCHED THEN
UPDATE SET
  target.customer_name = source.customer_name,
  target.city = source.city,
  target.product = source.product,
  target.quantity = source.quantity,
  target.price = source.price,
  target.order_status = source.order_status

WHEN NOT MATCHED THEN
INSERT (
  order_id,
  customer_name,
  city,
  product,
  quantity,
  price,
  order_status
)
VALUES (
  source.order_id,
  source.customer_name,
  source.city,
  source.product,
  source.quantity,
  source.price,
  source.order_status
);

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
2,2,0,0
